# NB_EXPORT: Tableau 시각화용 데이터 내보내기

**목적**: 모든 분석 결과를 Tableau가 읽을 수 있는 형식으로 변환·저장

**Tableau 입력 형식**:
- CSV: 포인트 데이터 (lat, lon + 속성) — Tableau에서 자동 지리 인식
- GeoJSON/Shapefile: 폴리곤 데이터 — Tableau 공간 파일 연결

**출력 디렉토리**: `03_visualization/tableau_data/`

## Tableau Parameter Toggle 구현 가이드

Tableau에서 5개 레이어를 On/Off 토글하려면:
1. **Parameter 생성**: Boolean 타입 5개 (P_Airspace, P_Obstacle, P_Noise, P_Terrain, P_Weather)
2. **Calculated Field**:
   ```
   [Dynamic Score] = 
   (IF [P_Airspace] THEN [score_airspace] ELSE 1 END)
   * (IF [P_Obstacle] THEN [score_obstacle] ELSE 1 END)
   * (IF [P_Noise] THEN [score_noise] ELSE 1 END)
   * (IF [P_Terrain] THEN [score_terrain] ELSE 1 END)
   * (IF [P_Weather] THEN [score_weather] ELSE 1 END)
   ```
3. **Color** hex grid by `[Dynamic Score]` → 토글 변경 시 맵 실시간 갱신

In [1]:
import pandas as pd
import geopandas as gpd
import numpy as np
from pathlib import Path
import shutil

BASE = Path(r"C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam")
OUT = BASE / "processed"
TAB = BASE / "03_visualization" / "tableau_data"
TAB.mkdir(parents=True, exist_ok=True)

print(f"Tableau 출력 디렉토리: {TAB}")

Tableau 출력 디렉토리: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data


## 1. 행정동 경계 (Shapefile)

Tableau는 GeoJSON보다 Shapefile을 더 안정적으로 지원

In [2]:
# 행정동 경계 → Shapefile
seongnam = gpd.read_file(OUT / "seongnam_boundary.gpkg", layer="dong")
seongnam_shp = TAB / "seongnam_dong"
seongnam_shp.mkdir(exist_ok=True)
seongnam.to_file(seongnam_shp / "seongnam_dong.shp", encoding="utf-8")
print(f"행정동 경계 Shapefile: {seongnam_shp} ({len(seongnam)} dongs)")

# 시 경계
city = gpd.read_file(OUT / "seongnam_boundary.gpkg", layer="city")
city.to_file(seongnam_shp / "seongnam_city.shp", encoding="utf-8")
print(f"시 경계 Shapefile 포함")

행정동 경계 Shapefile: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\seongnam_dong (50 dongs)
시 경계 Shapefile 포함


C:\Users\jimin\AppData\Local\Temp\ipykernel_50224\3393198758.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  seongnam.to_file(seongnam_shp / "seongnam_dong.shp", encoding="utf-8")
C:\Users\jimin\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'CSV_ADMI_CD' to 'CSV_ADMI_C'
  ogr_write(
C:\Users\jimin\AppData\Local\Temp\ipykernel_50224\3393198758.py:10: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  city.to_file(seongnam_shp / "seongnam_city.shp", encoding="utf-8")
C:\Users\jimin\anaconda3\Lib\site-packages\pyogrio\raw.py:733: RuntimeWarning: Normalized/laundered field name: 'CSV_ADMI_CD' to 'CSV_ADMI_C'
  ogr_write(


## 2. H3 그리드 + 점수 (CSV + GeoJSON)

핵심 데이터: 각 H3 셀의 좌표, urgency, 5개 제약 점수, composite score

In [3]:
# 제약 레이어 그리드
grid = gpd.read_file(OUT / "constraint_layers.gpkg")

# CSV (Tableau 포인트 데이터) — lat, lon으로 자동 인식
csv_cols = [
    "h3_index", "lat", "lon",
    "CSV_ADMI_CD", "ADM_NM", "GU_NM",
    "delivery_demand_index", "flow_pop_index", "urgency",
    "score_airspace", "score_obstacle", "score_noise",
    "score_terrain", "score_weather", "composite_score",
    "terrain_zone", "avg_slope",
    "tall_bldg_count", "total_bldg_count",
]
csv_cols = [c for c in csv_cols if c in grid.columns]
grid_csv = grid[csv_cols].copy()

# urgency 등급 (Tableau 색상 구간용)
grid_csv["urgency_level"] = pd.qcut(
    grid_csv["urgency"].rank(method="first"), 
    q=5, labels=["매우 낮음", "낮음", "보통", "높음", "매우 높음"]
)

# composite 등급
grid_csv["composite_level"] = np.where(
    grid_csv["composite_score"] == 0, "부적합",
    np.where(grid_csv["composite_score"] < 0.3, "낮음",
    np.where(grid_csv["composite_score"] < 0.6, "보통",
    np.where(grid_csv["composite_score"] < 0.8, "높음", "최적")))
)

grid_csv.to_csv(TAB / "grid_scores.csv", index=False, encoding="utf-8-sig")
print(f"그리드 CSV: {TAB / 'grid_scores.csv'} ({len(grid_csv)} rows)")

# GeoJSON (Tableau 공간 폴리곤용 — 헥사곤 형태 유지)
grid.to_file(TAB / "grid_hexagons.geojson", driver="GeoJSON")
print(f"그리드 GeoJSON: {TAB / 'grid_hexagons.geojson'}")

그리드 CSV: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\grid_scores.csv (1947 rows)


그리드 GeoJSON: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\grid_hexagons.geojson


## 3. 거점 정보 (CSV + GeoJSON)

In [4]:
# 최종 거점
hubs_path = OUT / "final_hubs.gpkg"
if hubs_path.exists():
    hubs = gpd.read_file(hubs_path, layer="hubs")
    
    # CSV
    hub_csv = hubs.drop(columns="geometry", errors="ignore").copy()
    hub_csv.to_csv(TAB / "final_hubs.csv", index=False, encoding="utf-8-sig")
    print(f"거점 CSV: {TAB / 'final_hubs.csv'} ({len(hub_csv)} hubs)")
    
    # GeoJSON
    hubs.to_file(TAB / "final_hubs.geojson", driver="GeoJSON")
    print(f"거점 GeoJSON: {TAB / 'final_hubs.geojson'}")
    
    # 서비스 권역
    try:
        service = gpd.read_file(hubs_path, layer="service_areas")
        service.to_file(TAB / "service_areas.geojson", driver="GeoJSON")
        print(f"서비스 권역 GeoJSON: {TAB / 'service_areas.geojson'}")
    except:
        print("서비스 권역 레이어 없음")
else:
    print("⚠ final_hubs.gpkg 미생성 — NB10 실행 필요")

거점 CSV: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\final_hubs.csv (4 hubs)
거점 GeoJSON: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\final_hubs.geojson
서비스 권역 GeoJSON: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\service_areas.geojson


## 4. 드론 경로 (GeoJSON)

In [5]:
routes_path = OUT / "drone_routes.gpkg"
if routes_path.exists():
    routes = gpd.read_file(routes_path)
    routes.to_file(TAB / "drone_routes.geojson", driver="GeoJSON")
    print(f"드론 경로 GeoJSON: {TAB / 'drone_routes.geojson'} ({len(routes)} routes)")
    
    # 경로 요약 CSV
    route_csv_path = OUT / "delivery_routes_summary.csv"
    if route_csv_path.exists():
        shutil.copy2(route_csv_path, TAB / "delivery_routes_summary.csv")
        print(f"경로 요약 CSV 복사 완료")
else:
    print("⚠ drone_routes.gpkg 미생성 — NB11 실행 필요")

드론 경로 GeoJSON: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\drone_routes.geojson (20 routes)
경로 요약 CSV 복사 완료


## 5. 비교 데이터 + 시계열 (CSV)

In [6]:
# 모드 비교
mode_files = ["mode_comparison.csv", "mode_radar_scores.csv"]
for fname in mode_files:
    src = OUT / fname
    if src.exists():
        shutil.copy2(src, TAB / fname)
        print(f"복사: {fname}")

# 시간별 수요 패턴 (NB04 출력)
hourly_path = OUT / "card_hourly_demand.parquet"
if hourly_path.exists():
    hourly = pd.read_parquet(hourly_path)
    hourly.to_csv(TAB / "hourly_demand.csv", index=False, encoding="utf-8-sig")
    print(f"시간별 수요 CSV: {len(hourly)} rows")

# 동별 경사도 통계
slope_path = OUT / "dong_slope_stats.csv"
if slope_path.exists():
    shutil.copy2(slope_path, TAB / "dong_slope_stats.csv")
    print(f"동별 경사도 CSV 복사 완료")

# 공공시설 전체 (거점 외 포함)
fac_path = OUT / "public_facilities.gpkg"
if fac_path.exists():
    fac = gpd.read_file(fac_path)
    fac_csv = fac.copy()
    fac_csv["lat"] = fac_csv.geometry.y
    fac_csv["lon"] = fac_csv.geometry.x
    fac_csv = fac_csv.drop(columns="geometry")
    fac_csv.to_csv(TAB / "public_facilities.csv", index=False, encoding="utf-8-sig")
    print(f"공공시설 CSV: {len(fac_csv)} facilities")

복사: mode_comparison.csv
복사: mode_radar_scores.csv
시간별 수요 CSV: 500 rows
공공시설 CSV: 172 facilities


## 6. 동별 종합 통계 (Tableau 조인용)

In [7]:
# 동별 집계 — Tableau에서 행정동 경계에 조인하여 코로플레스 맵 생성
if "ADM_NM" in grid.columns:
    dong_stats = grid.groupby(["ADM_NM", "GU_NM"]).agg(
        n_cells=("h3_index", "count"),
        avg_urgency=("urgency", "mean"),
        avg_composite=("composite_score", "mean"),
        avg_airspace=("score_airspace", "mean"),
        avg_obstacle=("score_obstacle", "mean"),
        avg_noise=("score_noise", "mean"),
        avg_terrain=("score_terrain", "mean"),
        avg_weather=("score_weather", "mean"),
        pct_feasible=("composite_score", lambda x: (x > 0).mean() * 100),
    ).round(3).reset_index()
    
    # 경사도 통계 병합
    if slope_path.exists():
        slopes = pd.read_csv(slope_path)
        dong_stats = dong_stats.merge(
            slopes[["ADM_NM", "avg_slope", "green_pct"]], 
            on="ADM_NM", how="left"
        )
    
    dong_stats.to_csv(TAB / "dong_summary.csv", index=False, encoding="utf-8-sig")
    print(f"동별 종합 통계 CSV: {TAB / 'dong_summary.csv'} ({len(dong_stats)} dongs)")
    print(dong_stats.head())

동별 종합 통계 CSV: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data\dong_summary.csv (50 dongs)
  ADM_NM GU_NM  n_cells  avg_urgency  avg_composite  avg_airspace  \
0    고등동   수정구      171        0.105          0.000         0.000   
1   구미1동   분당구       51        0.279          0.652         1.000   
2    구미동   분당구       51        0.423          0.550         1.000   
3    금곡동   분당구       80        0.285          0.277         0.462   
4   금광1동   중원구        8        0.114          0.000         0.000   

   avg_obstacle  avg_noise  avg_terrain  avg_weather  pct_feasible  
0         0.987      0.966          1.0          0.8          0.00  
1         0.882      0.914          1.0          0.8        100.00  
2         0.753      0.908          1.0          0.8        100.00  
3         0.872      0.934          1.0          0.8         46.25  
4         0.694      0.743          1.0          0.8          0.00  


## 7. 내보내기 결과 요약

In [8]:
import os

print("=" * 60)
print("Tableau 데이터 내보내기 완료")
print("=" * 60)
print(f"\n출력 디렉토리: {TAB}")
print(f"\n파일 목록:")

total_size = 0
for item in sorted(TAB.rglob("*")):
    if item.is_file():
        size_mb = item.stat().st_size / 1024 / 1024
        total_size += size_mb
        rel = item.relative_to(TAB)
        print(f"  {str(rel):<45} {size_mb:>6.2f} MB")

print(f"\n총 크기: {total_size:.1f} MB")
print(f"\n{'=' * 60}")
print("Tableau 설정 가이드:")
print("1. 새 데이터 소스 > 텍스트 파일 > grid_scores.csv")
print("2. 위도(lat), 경도(lon) 필드를 '지리적 역할' 지정")
print("3. 공간 파일 연결 > seongnam_dong.shp (행정동 경계)")
print("4. Parameter 5개 생성 (Boolean): P_Airspace ~ P_Weather")
print("5. Calculated Field: Dynamic Score (곱셈 공식)")
print("6. 이중 축 맵: 헥사곤 색상(Dynamic Score) + 행정동 경계")
print("=" * 60)

Tableau 데이터 내보내기 완료

출력 디렉토리: C:\Users\jimin\Desktop\1_BITAmin_16기\1_Seongnam\03_visualization\tableau_data

파일 목록:
  delivery_routes_summary.csv                     0.00 MB
  dong_summary.csv                                0.00 MB
  drone_routes.geojson                            0.01 MB
  final_hubs.csv                                  0.00 MB
  final_hubs.geojson                              0.00 MB
  grid_hexagons.geojson                           1.72 MB
  grid_scores.csv                                 0.40 MB
  hourly_demand.csv                               0.02 MB
  mode_comparison.csv                             0.00 MB
  mode_radar_scores.csv                           0.00 MB
  public_facilities.csv                           0.02 MB
  seongnam_dong\seongnam_city.cpg                 0.00 MB
  seongnam_dong\seongnam_city.dbf                 0.00 MB
  seongnam_dong\seongnam_city.prj                 0.00 MB
  seongnam_dong\seongnam_city.shp                 0.06 MB
  seongnam_don